# ex3 MNIST DNN

`objective_function.mnist_dnn` を読み込み、MNIST 分類モデルが学習可能かを確認するノートブックです。


In [1]:
import gzip
import importlib
import random
import struct
import urllib.request
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import objective_function.mnist_dnn as mnist_dnn

importlib.reload(mnist_dnn)

MNISTDNN = mnist_dnn.MNISTDNN
make_mnist_dnn = mnist_dnn.make_mnist_dnn

seed = 42
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
torch.set_default_dtype(dtype)

print(f"torch: {torch.__version__}")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")


torch: 2.5.1+cu121
device: cuda
gpu: Tesla T4


In [2]:
DATA_DIR = Path("data/mnist")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MNIST_URLS = {
    "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
    "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
    "test_images": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
    "test_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
}

def download(url: str, path: Path) -> None:
    if path.exists():
        print(f"cached: {path.name}")
        return
    print(f"downloading: {url}")
    urllib.request.urlretrieve(url, path)

def read_idx_images(path: Path) -> torch.Tensor:
    with gzip.open(path, "rb") as f:
        magic, num_images, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"unexpected image magic number: {magic}")
        data = torch.frombuffer(f.read(), dtype=torch.uint8).clone()
    return data.view(num_images, rows, cols).float() / 255.0

def read_idx_labels(path: Path) -> torch.Tensor:
    with gzip.open(path, "rb") as f:
        magic, num_items = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"unexpected label magic number: {magic}")
        labels = torch.frombuffer(f.read(), dtype=torch.uint8).clone()
    return labels.to(torch.long)

paths = {name: DATA_DIR / Path(url).name for name, url in MNIST_URLS.items()}
for name, url in MNIST_URLS.items():
    download(url, paths[name])

x_train = read_idx_images(paths["train_images"])
y_train = read_idx_labels(paths["train_labels"])
x_test = read_idx_images(paths["test_images"])
y_test = read_idx_labels(paths["test_labels"])

print("train:", x_train.shape, y_train.shape)
print("test :", x_test.shape, y_test.shape)


cached: train-images-idx3-ubyte.gz
cached: train-labels-idx1-ubyte.gz
cached: t10k-images-idx3-ubyte.gz
cached: t10k-labels-idx1-ubyte.gz


train: torch.Size([60000, 28, 28]) torch.Size([60000])
test : torch.Size([10000, 28, 28]) torch.Size([10000])


/tmp/ipykernel_130939/909995196.py:23: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:1560.)
  data = torch.frombuffer(f.read(), dtype=torch.uint8).clone()


In [3]:
train_subset_size = len(x_train)
test_subset_size = len(x_test)
batch_size = 128
learning_rate = 2e-3
epochs = 100
hidden_dims = (1024, 512, 256)
dropout_rate = 0.0

train_dataset = TensorDataset(x_train[:train_subset_size], y_train[:train_subset_size])
test_dataset = TensorDataset(x_test[:test_subset_size], y_test[:test_subset_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

model = make_mnist_dnn(hidden_dims=hidden_dims, dropout_rate=dropout_rate, dtype=dtype).to(device=device, dtype=dtype)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(model)
print(f"train subset: {len(train_dataset)}")
print(f"test subset: {len(test_dataset)}")


MNISTDNN(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=10, bias=True)
  )
)
train subset: 60000
test subset: 10000


In [4]:
def evaluate(model: torch.nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

    return total_loss / total, correct / total

def train_one_epoch(model: torch.nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


In [5]:
history = []

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_loss, test_acc = evaluate(model, test_loader)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })
    print(
        f"epoch {epoch:02d} "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4%} "
        f"test_loss={test_loss:.4f} test_acc={test_acc:.4%}"
    )

history[-1]


epoch 01 train_loss=0.2290 train_acc=93.0350% test_loss=0.1113 test_acc=96.6900%
epoch 02 train_loss=0.0905 train_acc=97.2733% test_loss=0.0795 test_acc=97.7000%
epoch 03 train_loss=0.0664 train_acc=97.9950% test_loss=0.0844 test_acc=97.6000%
epoch 04 train_loss=0.0507 train_acc=98.4417% test_loss=0.0719 test_acc=97.8800%
epoch 05 train_loss=0.0410 train_acc=98.7217% test_loss=0.0824 test_acc=97.6300%
epoch 06 train_loss=0.0351 train_acc=98.9150% test_loss=0.1004 test_acc=97.2000%
epoch 07 train_loss=0.0342 train_acc=98.9683% test_loss=0.0914 test_acc=97.8600%
epoch 08 train_loss=0.0277 train_acc=99.1367% test_loss=0.0919 test_acc=97.8800%
epoch 09 train_loss=0.0236 train_acc=99.2667% test_loss=0.0848 test_acc=98.0500%
epoch 10 train_loss=0.0216 train_acc=99.3767% test_loss=0.0876 test_acc=98.0200%
epoch 11 train_loss=0.0176 train_acc=99.4533% test_loss=0.0901 test_acc=97.8900%
epoch 12 train_loss=0.0191 train_acc=99.4433% test_loss=0.0895 test_acc=98.2300%
epoch 13 train_loss=0.0195 t

{'epoch': 100,
 'train_loss': 1.0351330804070357e-09,
 'train_acc': 1.0,
 'test_loss': 0.330361732673645,
 'test_acc': 0.9874}

In [6]:
final_train_loss, final_train_acc = evaluate(model, train_loader)
final_test_loss, final_test_acc = evaluate(model, test_loader)
print(f"final train loss: {final_train_loss:.6f}")
print(f"final train accuracy: {final_train_acc:.4%}")
print(f"final test loss: {final_test_loss:.6f}")
print(f"final test accuracy: {final_test_acc:.4%}")

sample_logits = model(x_test[:10].to(device))
sample_pred = sample_logits.argmax(dim=1).cpu()
print("pred :", sample_pred.tolist())
print("label:", y_test[:10].tolist())


final train loss: 0.000000
final train accuracy: 100.0000%
final test loss: 0.330362
final test accuracy: 98.7400%
pred : [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
label: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]


In [8]:
from common import local_rlct_estimater
from torch.nn.utils import parameters_to_vector

LocalRLCTTorchEstimator = local_rlct_estimater.LocalRLCTTorchEstimator

rlct_model = model.module if isinstance(model, torch.nn.DataParallel) else model
w0 = parameters_to_vector([param.detach() for param in rlct_model.parameters()]).detach()

def rlct_loss_fn(model: torch.nn.Module, images: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    logits = model(images)
    return criterion(logits, labels)

num_params = sum(param.numel() for param in rlct_model.parameters())
clip_radius = float(num_params ** 0.5)
print(f"num_params={num_params}, clip_radius={clip_radius:.6f}")

rlct_estimator = LocalRLCTTorchEstimator.from_tensors(
    rlct_model,
    rlct_loss_fn,
    x_train,
    y_train,
    w0=w0,
    device=device,
    dtype=dtype,
    eval_mode=True,
)

rlct_result = rlct_estimator.estimate(
    betas=[4, 8, 16, 32],
    step_size=5e-5,
    n_steps=2000,
    burn_in=100,
    thinning=10,
    clip_radius=3,
    grad_clip=75.0,
    n_chains=2,
    max_beta_step=0.05,
    regression_tail=None,
    use_weighted_regression=True,
    update_batch_size=batch_size,
    log_output_mode="step",
    seed=0,
)

print(f"estimated_lambda={rlct_result.lambda_hat:.6f}")
print(f"betaEf={rlct_result.betaEf}")
print(f"betaEf_se={rlct_result.betaEf_se}")
rlct_result.as_dict()


num_params=1462538, clip_radius=1209.354373
[LocalRLCT] beta=4 start (effective_step=5e-05, n_chains=2)
[LocalRLCT] beta=4 chain=1/2 step=100/2000 samples=0
[LocalRLCT] beta=4 chain=1/2 step=200/2000 samples=10
[LocalRLCT] beta=4 chain=1/2 step=300/2000 samples=20
[LocalRLCT] beta=4 chain=1/2 step=400/2000 samples=30
[LocalRLCT] beta=4 chain=1/2 step=500/2000 samples=40
[LocalRLCT] beta=4 chain=1/2 step=600/2000 samples=50
[LocalRLCT] beta=4 chain=1/2 step=700/2000 samples=60
[LocalRLCT] beta=4 chain=1/2 step=800/2000 samples=70
[LocalRLCT] beta=4 chain=1/2 step=900/2000 samples=80
[LocalRLCT] beta=4 chain=1/2 step=1000/2000 samples=90
[LocalRLCT] beta=4 chain=1/2 step=1100/2000 samples=100
[LocalRLCT] beta=4 chain=1/2 step=1200/2000 samples=110
[LocalRLCT] beta=4 chain=1/2 step=1300/2000 samples=120
[LocalRLCT] beta=4 chain=1/2 step=1400/2000 samples=130
[LocalRLCT] beta=4 chain=1/2 step=1500/2000 samples=140
[LocalRLCT] beta=4 chain=1/2 step=1600/2000 samples=150
[LocalRLCT] beta=4 c

{'lambda_hat': 5.423869814096951e-05,
 'betas': array([ 4.,  8., 16., 32.]),
 'betaEf': array([9.69484395e-06, 1.81549313e-05, 3.81670199e-05, 6.69780085e-05]),
 'mean_f': array([2.42371099e-06, 2.26936641e-06, 2.38543874e-06, 2.09306277e-06]),
 'ess_like_counts': array([380, 380, 380, 380]),
 'x0': array([ 0.02730495,  0.02964314, -0.00836687, ..., -0.37928832,
         0.93258619,  0.43579814], shape=(1462538,)),
 'objective_info': {'loss0': 8.324775868651102e-10,
  'scale': 60000.0,
  'dataset_size': 60000,
  'device': 'cuda',
  'dtype': 'torch.float32',
  'num_params': 1462538,
  'eval_mode': True,
  'sampler': 'sgld',
  'n_chains': 2,
  'base_step_size': 5e-05,
  'max_beta_step': 0.05,
  'update_batch_size': 128,
  'eval_max_batches': None,
  'replace_batches': True,
  'regression_tail': None,
  'use_weighted_regression': True,
  'log_output_mode': 'step',
  'log_every': 100,
  'regression_betas': array([ 4.,  8., 16., 32.])},
 'betaEf_std': array([1.93326287e-05, 3.73272623e-05, 